# Nocturne Aegis Cel Candidate Generator v3

This notebook generates LoRA candidate images with `OnomaAIResearch/Illustrious-xl-early-release-v0`.

v3 fixes the main issue from the previous run: **CLIP truncation**. SDXL still uses CLIP tokenizers with a practical 77-token limit per prompt channel. Long prompts get truncated and the model never sees most style instructions.

This version uses compact tag-style prompts, `prompt_2`, compact negative prompts, token length checks, and aspect-aware resolutions.


In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors pillow pandas tqdm

In [ ]:
import os, json, inspect, zipfile, gc
from pathlib import Path

import torch
import pandas as pd
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler, DPMSolverMultistepScheduler


In [ ]:
# =========================
# CONFIG
# =========================

MODEL_ID = "OnomaAIResearch/Illustrious-xl-early-release-v0"

PROMPTS_FILE = "prompts_illustrious_v3.json"
OUTPUT_ROOT = Path("outputs/nocturne_aegis_candidates_v3")
IMAGES_DIR = OUTPUT_ROOT / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# 10 prompts x 8 variations = 80 images.
NUM_IMAGES_PER_PROMPT = 8

# Good starting values for Illustrious/Kohaku-style SDXL.
NUM_INFERENCE_STEPS = 38
GUIDANCE_SCALE = 7.0
GUIDANCE_RESCALE = 0.0

# clip_skip=2 often helps anime checkpoints, but keep it easy to disable.
CLIP_SKIP = 2

# On T4 keep True. On L4/A100 you may set False for speed.
USE_CPU_OFFLOAD = True

# Scheduler: "euler_a" or "dpmpp_2m"
SCHEDULER = "euler_a"

BASE_SEED = 240504

# Aspect-aware sizes. SDXL likes dimensions divisible by 64.
ASPECT_SIZES = {
    "portrait": (896, 1152),
    "full_body": (832, 1216),
    "wide": (1216, 832),
    "square": (1024, 1024),
}

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Upload prompt file

Upload `prompts_illustrious_v3.json` to the Colab file browser before running the next cell. If it is missing, the notebook will create a fallback prompt file.


In [ ]:
if not Path(PROMPTS_FILE).exists():
    print(f"{PROMPTS_FILE} not found. Writing fallback prompt file.")
    fallback_prompts = [
  {
    "id": "nacel_001_portrait_woman",
    "aspect": "portrait",
    "content_tags": "adult woman, close portrait, long black hair, solemn gaze, dark ceremonial collar, looking at viewer",
    "caption": "nacel_v1, an adult woman with long black hair and a solemn gaze wearing a dark ceremonial collar, close portrait, looking at viewer"
  },
  {
    "id": "nacel_002_portrait_man",
    "aspect": "portrait",
    "content_tags": "adult man, close portrait, short silver hair, side lit face, armored coat collar, looking away",
    "caption": "nacel_v1, an adult man with short silver hair wearing an armored coat collar, close portrait, looking away"
  },
  {
    "id": "nacel_003_bust_pilot",
    "aspect": "portrait",
    "content_tags": "adult sky pilot, bust shot, cracked helmet in hand, long coat, cockpit glow, serious expression",
    "caption": "nacel_v1, an adult sky pilot holding a cracked helmet, wearing a long coat, bust shot, cockpit glow, serious expression"
  },
  {
    "id": "nacel_004_fullbody_spear",
    "aspect": "full_body",
    "content_tags": "adult spear bearer, full body, standing on cathedral bridge, layered armor, long ribbons, moon behind",
    "caption": "nacel_v1, an adult spear bearer standing on a cathedral bridge, full body, layered armor, long ribbons, moon in the background"
  },
  {
    "id": "nacel_005_fullbody_wanderer",
    "aspect": "full_body",
    "content_tags": "adult wanderer, full body, dark layered cloth, small metal accents, walking through flooded ruins",
    "caption": "nacel_v1, an adult wanderer wearing dark layered cloth with small metal accents, walking through flooded ruins, full body"
  },
  {
    "id": "nacel_006_action_swordsman",
    "aspect": "full_body",
    "content_tags": "adult swordsman, action pose, diagonal sword slash, broken armor plates, energy fragments, dynamic motion",
    "caption": "nacel_v1, an adult swordsman in an action pose with a diagonal sword slash, broken armor plates, energy fragments, dynamic motion"
  },
  {
    "id": "nacel_007_action_shield",
    "aspect": "square",
    "content_tags": "adult shield knight, impact pose, large ornate shield, shattered stone fragments, defensive stance",
    "caption": "nacel_v1, an adult shield knight holding a large ornate shield in an impact pose, shattered stone fragments, defensive stance"
  },
  {
    "id": "nacel_008_wide_city",
    "aspect": "wide",
    "content_tags": "lone figure, wide shot, ruined futuristic avenue, gothic towers, black water reflections, huge moon",
    "caption": "nacel_v1, a lone figure standing in a ruined futuristic avenue with gothic towers, black water reflections, huge moon, wide shot"
  },
  {
    "id": "nacel_009_mech",
    "aspect": "square",
    "content_tags": "winged humanoid machine, standing on launch rail, ornate armor plating, exposed joints, mechanical halo",
    "caption": "nacel_v1, a winged humanoid machine standing on a launch rail with ornate armor plating, exposed joints, and a mechanical halo"
  },
  {
    "id": "nacel_010_object_sword",
    "aspect": "square",
    "content_tags": "ornate sword and scabbard, object study, black metal blade, antique gold trim, violet gemstones",
    "caption": "nacel_v1, an ornate sword and scabbard with a black metal blade, antique gold trim, and violet gemstones, object study"
  }
]
    with open(PROMPTS_FILE, "w", encoding="utf-8") as f:
        json.dump(fallback_prompts, f, indent=2, ensure_ascii=False)

with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
    prompt_rows = json.load(f)

print("Loaded prompts:", len(prompt_rows))
prompt_rows[0]


## Prompt strategy

Do **not** use long prose prompts here. The previous prompt was 246 tokens and got truncated after 77 tokens.

`prompt` gets quality + content + short style. `prompt_2` gets compact global style.


In [ ]:
QUALITY_TAGS = (
    "masterpiece, best quality, amazing quality, very aesthetic, absurdres, "
    "full color, finished illustration, polished anime art, clean final render"
)

STYLE_SHORT = (
    "Nocturne Aegis Cel, 90s anime cel shading, modern digital painting, "
    "jewel tones, sharp lineart, glass eyes, dramatic rim light"
)

PROMPT_2_STYLE = (
    "finished full color anime illustration, elongated elegant proportions, "
    "sharp geometric face, melancholic expression, jewel tone palette, "
    "layered cel shadows, ambient occlusion, dramatic chiaroscuro, "
    "iridescent metal, matte cloth, clear silhouette"
)

NEGATIVE_PROMPT = (
    "lowres, worst quality, low quality, normal quality, sketch, rough sketch, "
    "pencil drawing, monochrome, grayscale, unfinished, draft, lineart only, "
    "bad anatomy, bad hands, extra fingers, missing fingers, malformed limbs, "
    "blurry, flat lighting, muddy colors, watermark, text, logo, signature"
)

NEGATIVE_PROMPT_2 = (
    "sketch, monochrome, grayscale, unfinished, bad anatomy, extra fingers, "
    "text, watermark, logo, blurry, noisy, cropped, jpeg artifacts"
)

def build_positive_prompt(content_tags: str) -> str:
    return f"{QUALITY_TAGS}, {content_tags}, {STYLE_SHORT}"


In [ ]:
# =========================
# LOAD MODEL
# =========================

torch_dtype = torch.float16

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    use_safetensors=True,
)

if SCHEDULER == "euler_a":
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
elif SCHEDULER == "dpmpp_2m":
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

# Suppress harmless BPE cleanup warning.
if hasattr(pipe, "tokenizer") and pipe.tokenizer is not None:
    pipe.tokenizer.clean_up_tokenization_spaces = False
if hasattr(pipe, "tokenizer_2") and pipe.tokenizer_2 is not None:
    pipe.tokenizer_2.clean_up_tokenization_spaces = False

if USE_CPU_OFFLOAD:
    pipe.enable_model_cpu_offload()
else:
    pipe.to("cuda")

print("Loaded:", MODEL_ID)
print("Scheduler:", pipe.scheduler.__class__.__name__)


In [ ]:
# =========================
# TOKEN LENGTH CHECKER
# =========================

MAX_TOKENS = 75

def token_count(text, tokenizer):
    ids = tokenizer(text, truncation=False, add_special_tokens=True).input_ids
    return len(ids)

def trim_csv_to_token_limit(text, tokenizer, max_tokens=MAX_TOKENS):
    tags = [t.strip() for t in text.split(",") if t.strip()]
    kept = []
    for tag in tags:
        candidate = ", ".join(kept + [tag])
        if token_count(candidate, tokenizer) <= max_tokens:
            kept.append(tag)
        else:
            break
    return ", ".join(kept)

def validate_and_trim(prompt, prompt_2, neg, neg_2):
    tok1 = pipe.tokenizer
    tok2 = pipe.tokenizer_2 if getattr(pipe, "tokenizer_2", None) is not None else pipe.tokenizer

    prompt = trim_csv_to_token_limit(prompt, tok1, MAX_TOKENS)
    prompt_2 = trim_csv_to_token_limit(prompt_2, tok2, MAX_TOKENS)
    neg = trim_csv_to_token_limit(neg, tok1, MAX_TOKENS)
    neg_2 = trim_csv_to_token_limit(neg_2, tok2, MAX_TOKENS)

    return prompt, prompt_2, neg, neg_2

sample_prompt = build_positive_prompt(prompt_rows[0]["content_tags"])
sample_prompt, sample_prompt_2, sample_neg, sample_neg_2 = validate_and_trim(
    sample_prompt, PROMPT_2_STYLE, NEGATIVE_PROMPT, NEGATIVE_PROMPT_2
)

print("prompt tokens:", token_count(sample_prompt, pipe.tokenizer), "/", 77)
print("prompt_2 tokens:", token_count(sample_prompt_2, pipe.tokenizer_2), "/", 77)
print("negative tokens:", token_count(sample_neg, pipe.tokenizer), "/", 77)
print("negative_2 tokens:", token_count(sample_neg_2, pipe.tokenizer_2), "/", 77)
print("
SMOKE PROMPT:
", sample_prompt)
print("
PROMPT_2:
", sample_prompt_2)


In [ ]:
# =========================
# SAFE PIPE CALL
# =========================

def call_pipe(prompt, prompt_2, negative_prompt, negative_prompt_2, width, height, seed):
    generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(seed)

    kwargs = dict(
        prompt=prompt,
        prompt_2=prompt_2,
        negative_prompt=negative_prompt,
        negative_prompt_2=negative_prompt_2,
        width=width,
        height=height,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
        original_size=(width, height),
        target_size=(width, height),
        crops_coords_top_left=(0, 0),
    )

    sig = inspect.signature(pipe.__call__).parameters

    if "clip_skip" in sig and CLIP_SKIP is not None:
        kwargs["clip_skip"] = CLIP_SKIP

    if "guidance_rescale" in sig:
        kwargs["guidance_rescale"] = GUIDANCE_RESCALE

    with torch.inference_mode():
        return pipe(**kwargs).images[0]


## Smoke test

Run this before the full batch. If the result is still sketchy, raise `GUIDANCE_SCALE` to 7.5 and keep the prompt compact.


In [ ]:
smoke_row = prompt_rows[2] if len(prompt_rows) > 2 else prompt_rows[0]
width, height = ASPECT_SIZES.get(smoke_row.get("aspect", "square"), (1024, 1024))

positive = build_positive_prompt(smoke_row["content_tags"])
positive, prompt_2, neg, neg_2 = validate_and_trim(
    positive, PROMPT_2_STYLE, NEGATIVE_PROMPT, NEGATIVE_PROMPT_2
)

print("prompt tokens:", token_count(positive, pipe.tokenizer))
print("prompt_2 tokens:", token_count(prompt_2, pipe.tokenizer_2))
print("WIDTH/HEIGHT:", width, "x", height)
print("
PROMPT:
", positive)

img = call_pipe(
    prompt=positive,
    prompt_2=prompt_2,
    negative_prompt=neg,
    negative_prompt_2=neg_2,
    width=width,
    height=height,
    seed=BASE_SEED,
)
img


## Full generation: 10 prompts x 8 seeds = 80 candidates

In [ ]:
metadata = []

for p_idx, row in enumerate(tqdm(prompt_rows, desc="Prompts")):
    aspect = row.get("aspect", "square")
    width, height = ASPECT_SIZES.get(aspect, (1024, 1024))

    base_positive = build_positive_prompt(row["content_tags"])
    positive, prompt_2, neg, neg_2 = validate_and_trim(
        base_positive, PROMPT_2_STYLE, NEGATIVE_PROMPT, NEGATIVE_PROMPT_2
    )

    for v_idx in tqdm(range(NUM_IMAGES_PER_PROMPT), desc=row["id"], leave=False):
        seed = BASE_SEED + (p_idx * 1000) + v_idx
        image = call_pipe(
            prompt=positive,
            prompt_2=prompt_2,
            negative_prompt=neg,
            negative_prompt_2=neg_2,
            width=width,
            height=height,
            seed=seed,
        )

        stem = f"{row['id']}_seed{seed}"
        image_path = IMAGES_DIR / f"{stem}.png"
        caption_path = IMAGES_DIR / f"{stem}.txt"

        image.save(image_path)
        with open(caption_path, "w", encoding="utf-8") as f:
            f.write(row["caption"].strip() + "
")

        metadata.append({
            "id": row["id"],
            "seed": seed,
            "aspect": aspect,
            "width": width,
            "height": height,
            "image": str(image_path),
            "caption_file": str(caption_path),
            "caption": row["caption"],
            "prompt": positive,
            "prompt_2": prompt_2,
            "negative_prompt": neg,
            "negative_prompt_2": neg_2,
            "prompt_tokens": token_count(positive, pipe.tokenizer),
            "prompt_2_tokens": token_count(prompt_2, pipe.tokenizer_2),
        })

        display(image.resize((min(320, image.width), int(image.height * min(320, image.width) / image.width))))

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df = pd.DataFrame(metadata)
df.to_csv(OUTPUT_ROOT / "metadata.csv", index=False)
df.to_json(OUTPUT_ROOT / "metadata.jsonl", orient="records", lines=True, force_ascii=False)

df.head()


In [ ]:
# =========================
# CONTACT SHEET
# =========================

def make_contact_sheet(image_paths, out_path, thumb_w=220, label_h=32, cols=5):
    thumbs = []
    for path in image_paths:
        img = Image.open(path).convert("RGB")
        ratio = img.height / img.width
        thumb_h = int(thumb_w * ratio)
        img = img.resize((thumb_w, thumb_h))
        canvas = Image.new("RGB", (thumb_w, thumb_h + label_h), "white")
        canvas.paste(img, (0, 0))
        draw = ImageDraw.Draw(canvas)
        draw.text((6, thumb_h + 8), Path(path).stem[-18:], fill=(0, 0, 0))
        thumbs.append(canvas)

    rows = (len(thumbs) + cols - 1) // cols
    cell_h = max(t.height for t in thumbs)
    sheet = Image.new("RGB", (cols * thumb_w, rows * cell_h), "white")

    for i, thumb in enumerate(thumbs):
        x = (i % cols) * thumb_w
        y = (i // cols) * cell_h
        sheet.paste(thumb, (x, y))

    sheet.save(out_path, quality=92)
    return sheet

image_paths = sorted(str(p) for p in IMAGES_DIR.glob("*.png"))
contact_path = OUTPUT_ROOT / "contact_sheet.jpg"
sheet = make_contact_sheet(image_paths, contact_path)
print("Saved:", contact_path)
sheet


In [ ]:
# =========================
# ZIP OUTPUT
# =========================

archive_path = OUTPUT_ROOT / "output_archive_v3.zip"

with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file() and file_path != archive_path:
            z.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))

print("Archive:", archive_path)
print("Files:", len(list(OUTPUT_ROOT.rglob('*'))))
